# Gemini finetune test

In [5]:
# Read parallel sentences file
import os
import json

filename = "data_finetune\\kalamang_rule\\kalamang_all_batch_prompts_ADJ_400_openai_batch_results_20_subset.json"

if os.path.exists(filename):
    with open(filename, "r", encoding="utf-8") as file:
        data = json.load(file)
        for entry in data:
            print(f"Source: {entry['generated_sentence']}")
            print(f"Target: {entry['generated_eng_translation']}")
    print(f"Loaded {len(data)} entries from {filename}")

else:
    print(f"File {filename} does not exist.")

Source: wane koraru dog iam fish=obj bite
Target: The proximal dog has bitten the fish.
Source: bal se siktaktak koraru dog iam fish=obj bite
Target: The slow dog has bitten the fish.
Source: keit dog iam fish=obj bite
Target: The top dog has bitten the fish.
Source: bal se sor=at koraru dog iam garos fish=obj bite.
Target: The dog has bitten the low fish.
Source: bal se sor=at koraru dog iam fish=obj bite
Target: The dog has bitten the fish.
Source: bal se sor=at koraru dog iam kalawen fish=obj bite
Target: The dog has bitten the soft fish.
Source: bal se sor=at koraru dog iam fish=obj bite
Target: The dog has bitten the fish.
Source: bal se sor=at na=koraru dog iam fish=obj
Target: The dog causes the fish to be bitten.
Source: watko se sor=at koraru fish iam bite
Target: The fish has been bitten at the proximal location.
Source: bal se sor=at koraru kinkinun dog iam fish=obj bite
Target: The small dog has bitten the fish.
Source: bal se susa iam fish=obj bite
Target: The dog has bitt

In [6]:
# Clean sentences (Keep all entries within quotes ‘’), if not, keep the whole
cleaned_data = []
for entry in data:
    original = entry['generated_sentence']
    translated = entry['generated_eng_translation']

    # Extract text within quotes if present
    if '‘' in original and '’' in original:
        start = original.index('‘') + 1
        end = original.index('’')
        original = original[start:end].strip()
    
    if '‘' in translated and '’' in translated:
        start = translated.index('‘') + 1
        end = translated.index('’')
        translated = translated[start:end].strip()

    # If "Translation: " prefix exists, remove it
    if translated.lower().startswith("translation:"):
        translated = translated[len("translation:"):].strip()
    
    if translated.lower().startswith("translation -"):
        translated = translated[len("translation -"):].strip()
    if translated.lower().startswith("translation in english:"):
        translated = translated[len("translation in english:"):].strip()

    # Do it for all other types of quotes as well
    quote_pairs = [('“', '”'), ('‘', '’'), ('"', '"'), ("'", "'")]
    for open_quote, close_quote in quote_pairs:
        if translated.startswith(open_quote) and translated.endswith(close_quote):
            translated = translated[1:-1].strip()
        if original.startswith(open_quote) and original.endswith(close_quote):
            original = original[1:-1].strip()
    cleaned_data.append({
        'original_sentence': original,
        'translated_sentence': translated
    })

cleaned_data

[{'original_sentence': 'wane koraru dog iam fish=obj bite',
  'translated_sentence': 'The proximal dog has bitten the fish.'},
 {'original_sentence': 'bal se siktaktak koraru dog iam fish=obj bite',
  'translated_sentence': 'The slow dog has bitten the fish.'},
 {'original_sentence': 'keit dog iam fish=obj bite',
  'translated_sentence': 'The top dog has bitten the fish.'},
 {'original_sentence': 'bal se sor=at koraru dog iam garos fish=obj bite.',
  'translated_sentence': 'The dog has bitten the low fish.'},
 {'original_sentence': 'bal se sor=at koraru dog iam fish=obj bite',
  'translated_sentence': 'The dog has bitten the fish.'},
 {'original_sentence': 'bal se sor=at koraru dog iam kalawen fish=obj bite',
  'translated_sentence': 'The dog has bitten the soft fish.'},
 {'original_sentence': 'bal se sor=at koraru dog iam fish=obj bite',
  'translated_sentence': 'The dog has bitten the fish.'},
 {'original_sentence': 'bal se sor=at na=koraru dog iam fish=obj',
  'translated_sentence':

In [7]:
data = cleaned_data

In [8]:
import random
train_test_split = 0.8

# Randomly shuffle data
random.shuffle(data)

train_size = int(len(data) * train_test_split)
train_data = data[:train_size]
test_data = data[train_size:]

len(train_data), len(test_data)

(2523, 631)

In [9]:
test_data

[{'original_sentence': 'muisese gala tiri',
  'translated_sentence': 'hungry spear drum'},
 {'original_sentence': 'Mu garos muap=nin.',
  'translated_sentence': 'We are not low yet.'},
 {'original_sentence': 'ka-mun tok bo=in ariemun muawese eba metko bo=te',
  'translated_sentence': 'Don’t you go yet, after Friday has passed, go hungry!'},
 {'original_sentence': 'Emun se... kasko wanir. emun se kasko wanir.',
  'translated_sentence': 'Her mother [did it] already... already clean behind.'},
 {'original_sentence': 'tumun se bo temun mia-rip child iam go nares dist-dgr',
  'translated_sentence': 'The child has become too heavy.'},
 {'original_sentence': 'bal=nan kinkinun dalang=i pasier=ko yie marua',
  'translated_sentence': 'The small dog also jumped in the sea and swam away from the shore.'},
 {'original_sentence': 'ma [ ... ] bo amdir=ka bo muap mor - ruo',
  'translated_sentence': 'She goes to the garden to dig up sour food.'},
 {'original_sentence': 'kinkinun pɛr tep',
  'translate

In [10]:
# Modify into gemini finetune format
output_filename = "train.jsonl"
output_path = "gemini_finetune_data"

target_language = "kalamang"


combined_output_path = os.path.join(output_path, target_language)
os.makedirs(combined_output_path, exist_ok=True)
output_filename = os.path.join(combined_output_path, output_filename)


with open(output_filename, "w", encoding="utf-8") as outfile:
    for entry in train_data:
        json_line = {
            "contents":[
                {
                    "role": "user",
                    "parts": [
                        {"text": f"Translate to english from {target_language}: " + entry["original_sentence"]}
                    ]
                },
                {

                    "role": "model",
                    "parts": [
                        {"text": entry["translated_sentence"]}
                    ],
                }
            ]
        }
        outfile.write(json.dumps(json_line) + "\n")

print(f"Saved {len(train_data)} training entries to {output_filename}")

output_filename = os.path.join(combined_output_path, "test.jsonl")
with open(output_filename, "w", encoding="utf-8") as outfile:
    for entry in test_data:
        json_line = {
            "contents":[
                {
                    "role": "user",
                    "parts": [
                        {"text": f"Translate to english from {target_language}: " + entry["original_sentence"]}
                    ]
                },
                {

                    "role": "model",
                    "parts": [
                        {"text": entry["translated_sentence"]}
                    ],
                }
            ]
        }
        outfile.write(json.dumps(json_line) + "\n")

print(f"Saved {len(test_data)} testing entries to {output_filename}")


Saved 2523 training entries to gemini_finetune_data\kalamang\train.jsonl
Saved 631 testing entries to gemini_finetune_data\kalamang\test.jsonl


In [13]:
# Create test set using original parallel sentences after 400

parallel_sents_file = "parallel_sents/kalamang_parallel_sentences_with_pos_and_unimorph_cleaned.json"

if os.path.exists(parallel_sents_file):
    with open(parallel_sents_file, "r", encoding="utf-8") as file:
        parallel_data = json.load(file)
    test_set = parallel_data[400:]  # Assuming first 400 were used for training
    print(f"Created test set with {len(test_set)} entries from parallel sentences.")
else:
    print(f"File {parallel_sents_file} does not exist.")

Created test set with 521 entries from parallel sentences.


In [15]:
test_set_sents_only = [
    {
        "original_sentence": entry["sentence"],
        "translated_sentence": entry["translated_sentence_clean"]
    }
    for entry in test_set
]

In [16]:
test_set_sents_only[0]

{'original_sentence': 'A: bir-na =teba eh ‘[They are] drinking beer, right?’ B: bir-na =teba ‘[They are] drinking beer.’ A: mier=a bir-na kona bir=at na ‘They are drinking beer, look, drinking beer.’ B: mier bir-na =teba',
 'translated_sentence': 'They are drinking beer.'}

In [17]:
# save as test.jsonl
output_filename = os.path.join(combined_output_path, "test_parallel.jsonl")
with open(output_filename, "w", encoding="utf-8") as outfile:  
    for entry in test_set_sents_only:
        json_line = {
            "contents":[
                {
                    "role": "user",
                    "parts": [
                        {"text": f"Translate to english from {target_language}: " + entry["original_sentence"]}
                    ]
                },
                {

                    "role": "model",
                    "parts": [
                        {"text": entry["translated_sentence"]}
                    ],
                }
            ]
        }
        outfile.write(json.dumps(json_line) + "\n")
print(f"Saved {len(test_set_sents_only)} testing entries to {output_filename}")





Saved 521 testing entries to gemini_finetune_data\kalamang\test_parallel.jsonl
